# Classes e objetos

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Escrever uma classe com campos privados, um construtor e métodos, e criar objetos a partir dela.
- Usar um record para dados simples e um enum para um conjunto fixo de valores.
- Compartilhar comportamento com herança e interfaces, e sobrescrever métodos como toString.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma classe descreve um tipo de objeto
2. Construtores: preparando um objeto
3. Variáveis guardam referências
4. private: protegendo os campos
5. toString: como um objeto é impresso
6. Records: classes para dados simples
7. Enums: um conjunto fixo de valores
8. Interfaces e herança

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Uma classe descreve um tipo de objeto

Até aqui os seus valores eram números e texto. Uma **classe** permite criar o seu próprio tipo de valor. Ela descreve o que cada objeto **sabe** (os **campos**, variáveis dentro dele) e o que ele **faz** (os **métodos**). O `new` cria um objeto a partir da classe, e cada objeto tem a própria cópia dos campos. Use um ponto para alcançá-los: `a.count`, `a.add()`.

In [ ]:
class Counter {
    int count;          // um campo: cada Counter tem o seu

    void add() {        // um método: trabalha com o campo deste objeto
        count++;
    }
}

void main() {
    Counter a = new Counter();
    Counter b = new Counter();
    a.add();
    a.add();
    b.add();
    IO.println("a: " + a.count + ", b: " + b.count);
}

🧪 **Experimente:** acrescente um método `void reset()` que volta `count` para 0, e chame-o só em `a`.

## 2. Construtores: preparando um objeto

Um **construtor** roda quando o `new` cria o objeto, para dar os primeiros valores aos campos. Ele tem o nome da classe e nenhum tipo de retorno. Lá dentro, `this.name` é o campo, enquanto `name` sozinho é o parâmetro: `this.name = name;` copia um para o outro.

In [ ]:
class Pet {
    String name;
    int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    String describe() {
        return name + " (" + age + (age == 1 ? " ano)" : " anos)");
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    Pet mia = new Pet("Mia", 1);
    IO.println(rex.describe());
    IO.println(mia.describe());
}

🧪 **Experimente:** tire o `this.` de `this.name = name;` e rode: o campo fica `null`, porque você copiou o parâmetro nele mesmo. Depois tente `new Pet("Bob")` com um argumento só.

## 3. Variáveis guardam referências

Um objeto mora no heap, e a variável guarda uma seta para ele, como nos arrays. Dois `new` criam dois objetos, mesmo com valores iguais. `Seat s3 = s1;` não cria objeto nenhum: as duas variáveis apontam para o mesmo. Por isso o `==` pergunta "o mesmo objeto?", enquanto o `equals` pergunta "os mesmos valores?". Os slides mostram isso passo a passo.

#### 🎞️ Objetos, referências e igualdade

![Dois news criam dois objetos mesmo com valores iguais; atribuir compartilha um objeto; == compara objetos e equals compara valores](objects-memory-steps.pt-br.svg)

In [ ]:
record Seat(char row, int number) {}

void main() {
    Seat s1 = new Seat('A', 1);
    Seat s2 = new Seat('A', 1);
    Seat s3 = s1;
    IO.println(s1 == s2);
    IO.println(s1.equals(s2));
    IO.println(s1 == s3);
}

🧪 **Experimente:** preveja as três linhas antes de rodar. (O `record` vem no passo 6: ele escreve o `equals` para você.)

### ✏️ Exercício: Mude um objeto por um método

Um método recebe uma cópia da referência, não do objeto. Faça `fill` pôr v na caixa que recebe: o Box de quem chamou muda também.

In [ ]:
class Box {
    int value = -1;
}

/** Fills a new box with v, then reads it back from the caller's side. */
int filledWith(int v) {
    Box mine = new Box();
    fill(mine, v);
    return mine.value;
}

void fill(Box box, int v) {
    // TODO: mude box.value
}

void main() {
    check("quem chamou vê 7", filledWith(7), 7);
    check("e depois 0", filledWith(0), 0);
}

<details>
<summary>Ver uma resposta</summary>

```java
class Box {
    int value = -1;
}

/** Fills a new box with v, then reads it back from the caller's side. */
int filledWith(int v) {
    Box mine = new Box();
    fill(mine, v);
    return mine.value;
}

void fill(Box box, int v) {
    box.value = v;
}

void main() {
    check("quem chamou vê 7", filledWith(7), 7);
    check("e depois 0", filledWith(0), 0);
}
```

</details>

## 4. private: protegendo os campos

Se qualquer código pode mudar um campo, nada impede um valor inválido, como um termostato a 500 graus. Marque os campos como `private`: só os métodos da própria classe podem mexer neles. O resto do código passa por métodos que conferem os valores. Isso se chama **encapsulamento**.

In [ ]:
class Thermostat {
    private int target = 20;

    void setTarget(int degrees) {
        if (degrees < 10 || degrees > 30) {
            IO.println("Ignorado: " + degrees + " está fora da faixa");
            return;
        }
        target = degrees;
    }

    int getTarget() {
        return target;
    }
}

void main() {
    Thermostat t = new Thermostat();
    t.setTarget(23);
    t.setTarget(500);
    IO.println("Alvo: " + t.getTarget());
}

🧪 **Experimente:** acrescente um método `void warmer()` que sobe o alvo em 1 grau, mas nunca acima de 30.

## 5. toString: como um objeto é impresso

Imprimir um objeto chama o método `toString()` dele. Sem ele você vê o nome da classe e um código, como `Pet@1b6d3586`. Escreva o seu para escolher o texto. O `@Override` avisa o compilador que você está substituindo um método que toda classe herda, e ele reclama se o nome estiver errado.

In [ ]:
class Pet {
    private final String name;
    private final int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    @Override
    public String toString() {
        return name + ", " + age;
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    IO.println(rex);
    IO.println("Meu pet: " + rex);
}

🧪 **Experimente:** apague o método `toString` e rode: esse é o texto padrão. Depois escreva errado, `tostring`, com o `@Override` ainda em cima, e leia o erro.

## 6. Records: classes para dados simples

Muitas classes só guardam dados. Um **record** escreve todo o código de costume para você a partir de uma linha: `record Song(String title, int seconds) {}` dá um construtor, os acessores `title()` e `seconds()`, o `toString` e um `equals` que compara os valores. Os campos dele não mudam depois da criação. Ainda dá para acrescentar métodos dentro das chaves.

In [ ]:
record Song(String title, int seconds) {
    String length() {
        return seconds / 60 + ":" + String.format("%02d", seconds % 60);
    }
}

void main() {
    Song intro = new Song("Intro", 95);
    IO.println(intro);
    IO.println(intro.title() + " dura " + intro.length());
    IO.println(intro.equals(new Song("Intro", 95)));
}

🧪 **Experimente:** acrescente um campo `String artist` ao record e conserte as chamadas `new Song(...)`. Depois tente `intro.seconds = 10;`: records não mudam.

### ✏️ Exercício: A etiqueta de um objeto

Um Seat tem uma letra de fila e um número. Devolva a etiqueta dele, como `A1`, a partir dos acessores do record.

In [ ]:
record Seat(char row, int number) {}

String label(Seat seat) {
    // TODO: junte seat.row() e seat.number()
    return "";
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}

<details>
<summary>Ver uma resposta</summary>

```java
record Seat(char row, int number) {}

String label(Seat seat) {
    return "" + seat.row() + seat.number();
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}
```

</details>

## 7. Enums: um conjunto fixo de valores

Quando um valor só pode ser uma entre poucas opções (os dias da semana, os naipes de um baralho), um **enum** as lista. Aí o compilador recusa qualquer outra coisa, e um `switch` pode cobrir cada opção. `values()` devolve todas, em ordem.

In [ ]:
enum Direction { NORTH, EAST, SOUTH, WEST }

Direction turnRight(Direction d) {
    return switch (d) {
        case NORTH -> Direction.EAST;
        case EAST -> Direction.SOUTH;
        case SOUTH -> Direction.WEST;
        case WEST -> Direction.NORTH;
    };
}

void main() {
    Direction facing = Direction.NORTH;
    facing = turnRight(facing);
    IO.println("Agora virado para " + facing);
    for (Direction d : Direction.values()) {
        IO.print(d + " ");
    }
    IO.println();
}

🧪 **Experimente:** vire à direita mais três vezes. Depois apague um `case` e leia o erro: o switch não cobre mais todas as direções (e ele não precisa de `default`, porque o enum só tem essas quatro).

## 8. Interfaces e herança

Uma **interface** lista métodos que várias classes prometem ter, para que o código possa usar qualquer uma delas do mesmo jeito. Cada classe que a `implements` escreve a própria versão. Uma classe também pode `extends` (estender) outra para herdar os campos e métodos dela e mudar alguns com `@Override`.

In [ ]:
interface Discount {
    double apply(double price);
}

class Percent implements Discount {
    private final int percent;

    Percent(int percent) {
        this.percent = percent;
    }

    public double apply(double price) {
        return price * (100 - percent) / 100;
    }
}

class Fixed implements Discount {
    public double apply(double price) {
        return Math.max(0, price - 10);
    }
}

void main() {
    List<Discount> offers = List.of(new Percent(20), new Fixed());
    for (Discount d : offers) {
        IO.println(d.getClass().getSimpleName() + ": " + d.apply(30));
    }
}

🧪 **Experimente:** escreva um terceiro desconto, `HalfPrice`, e ponha na lista: o laço o usa sem nenhuma outra mudança.

## Em resumo

Uma **classe** descreve um tipo de objeto: seus **campos** (o que ele sabe) e seus **métodos** (o que ele faz). `new` cria um objeto, e o **construtor** o prepara. Deixe os campos `private`, para o resto do código passar pelos métodos.

```java
record Song(String title, int seconds) {}

class Playlist {
    private final List<Song> songs = new ArrayList<>();

void add(Song song) {
        songs.add(song);
    }

int totalSeconds() {
        int total = 0;
        for (Song s : songs) {
            total += s.seconds();
        }
        return total;
    }
}

void main() {
    Playlist mix = new Playlist();
    mix.add(new Song("Intro", 95));
    mix.add(new Song("Tema", 210));
    IO.println(mix.totalSeconds() + " segundos");
    IO.println(new Song("Intro", 95));
}
```

Um **record** escreve para você o construtor, os getters, o `toString` e o `equals`. Um `enum` lista valores fixos. Uma classe pode estender (`extends`) outra ou implementar (`implements`) uma interface, e `@Override` substitui um método herdado.

**Cuidado:** dois objetos com os mesmos valores não são `==`. Compare-os com `equals`: records já o têm; numa classe você escreve o `equals` (e o `hashCode`).

## Leituras

- [Classes and Objects in Java](https://www.freecodecamp.org/news/the-java-handbook/#heading-classes-and-objects-in-java) (freeCodeCamp, The Java Handbook, 15 min): Classes e objetos, métodos, construtores, getters e setters, depois herança e sobrescrita de métodos, para Formas, Animais e Folha de pagamento. O livro escreve System.out.println dentro de public static void main; no Sphinx, IO.println e void main() fazem o mesmo.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Sua primeira classe** · Fácil
- **Classe Retângulo** · Fácil
- **Livro com toString()** · Fácil
- **Pontos como records** · Fácil
- **Gerador de IDs (static)** · Fácil
- **Conta bancária (encapsulamento)** · Médio
- **Moedas (enum)** · Médio
- **Formas (herança)** · Médio
- **Animais (interfaces)** · Médio
- **Pontos iguais (equals e hashCode)** · Difícil
- **Folha de pagamento (polimorfismo)** · Difícil
- Teste: **Sistema da biblioteca** · Difícil